Common to all humans (baseline, every human role gets these regardless): internal_wiki, team_calendar, project_tracker

Human roles — role-specific additions on top of common:

Role	Resources
Finance	payroll_db, financial_reports, vendor_contracts
Engineering	source_code_repo, deployment_pipeline, shared_drive_general
IT/Admin	admin_credentials_vault, security_audit_logs, deployment_pipeline, employee_directory
HR	employee_directory, payroll_db, legal_case_files
Sales	customer_support_tickets, vendor_contracts, analytics_dashboard

Non-human functions (no "common" baseline — each function gets only what its job needs, no implicit shared access):

Function	Resources
data-pipeline	analytics_dashboard, customer_pii_store, shared_drive_general
backup-automation	shared_drive_general, security_audit_logs
monitoring	security_audit_logs, analytics_dashboard
ci-cd	source_code_repo, deployment_pipeline
security-scanning	security_audit_logs, admin_credentials_vault, source_code_repo
ai-assistant	internal_wiki, customer_support_tickets, analytics_dashboard
integration	shared_drive_general, analytics_dashboard, customer_pii_store


Human: Junior excluded from high; Senior/Manager uncapped

Non-human: Fully-autonomous excluded from high; Semi-autonomous excluded from 3 named high resources; Supervised uncapped


If resource sensitivity is high:

granted_by = a randomly chosen Manager-tier entity — same role as grantee if human, any Engineering Manager if non-human
expires_at = granted_at + 90 days

If resource sensitivity is low or medium:

granted_by = "IT_ADMIN_SYSTEM"
expires_at = null

In [0]:
import uuid
import datetime
import random
from pyspark.sql.functions import col

In [0]:
# Human common resources
human_common_resources = ["internal_wiki", "team_calendar", "project_tracker"]

# Human role resources mapping
human_role_resources = {
    "Finance": ["payroll_db", "financial_reports", "vendor_contracts"],
    "Engineering": ["source_code_repo", "deployment_pipeline", "shared_drive_general"],
    "IT": ["admin_credentials_vault", "security_audit_logs", "deployment_pipeline", "employee_directory"],
    "HR": ["employee_directory", "payroll_db", "legal_case_files"],
    "Sales": ["customer_pii_store", "customer_support_tickets", "vendor_contracts", "analytics_dashboard"],
}

# Non-human role resources mapping
non_human_role_resources = {
    "data-pipeline": ["analytics_dashboard", "customer_pii_store", "shared_drive_general"],
    "backup-automation": ["shared_drive_general", "security_audit_logs"],
    "monitoring": ["security_audit_logs", "analytics_dashboard"],
    "ci-cd": ["source_code_repo", "deployment_pipeline"],
    "security-scanning": ["security_audit_logs", "admin_credentials_vault", "source_code_repo"],
    "ai-assistant": ["internal_wiki", "customer_support_tickets", "analytics_dashboard"],
    "integration": ["shared_drive_general", "analytics_dashboard", "customer_pii_store"],
}

# Very high risk resources
very_high_risk_resources = ["admin_credentials_vault", "security_audit_logs", "legal_case_files"]

In [0]:
entities_df = spark.read.table("entity_risk_platform.seed_data.entities").select("entity_id", "entity_type", "role", "tier")

resources_df = spark.read.table("entity_risk_platform.seed_data.resources")

In [0]:
# Manager dict
manager_dict = {}
for row in entities_df.filter(col("tier") == "Manager").collect():
    manager_dict.setdefault(row["role"], []).append(row["entity_id"])    

# Resource dict
resource_dict = { row["resource_name"]: (row["resource_id"], row["resource_sensitivity"]) for row in resources_df.collect() }

high_sensitive_resources = [r_name for r_name, (r_id, r_sensitivity) in resource_dict.items() if r_sensitivity == "high"]

very_high_sensitive_resources = [r_name for r_name in resource_dict if r_name in very_high_risk_resources]

In [0]:
new_grant_list = []

In [0]:
def generate_grant(entity_id, resource_id, granter, grant_date, expiry_date):
  grant_id = str(uuid.uuid4())
  return {
    "grant_id": grant_id,
    "entity_id": entity_id,
    "resource_id": resource_id,
    "granted_by": granter, 
    "granted_at": grant_date,
    "expires_at": expiry_date
  } 

In [0]:
def generate_grant_list(resources, role, entity):
    for resource in resources:
      r_id, r_sensitivity = resource_dict[resource]
      if r_sensitivity == "high":
        manager_ids = manager_dict.get(role, [])
        granter = random.choice(manager_ids) if manager_ids else "SYSTEM"
        now = datetime.datetime.now()
        new_grant_list.append(generate_grant(entity, r_id, granter, now, now + datetime.timedelta(days=90)))
      else:
        new_grant_list.append(generate_grant(entity, r_id, "SYSTEM", datetime.datetime.now(), None))

In [0]:
for entity in entities_df.collect():
  row = entity.asDict()
  if row["entity_type"] == "human":
    human_resources = human_common_resources + human_role_resources[row["role"]]
    if row["tier"] == "Junior" :
      human_resources = [hr for hr in human_resources if hr not in high_sensitive_resources]
    generate_grant_list(human_resources, row["role"], row["entity_id"])
  if row["entity_type"] in ("service_account", "agent") :
    non_human_resources = non_human_role_resources[row["role"]]
    if row["tier"] == "Fully-autonomous":
      non_human_resources = [nhr for nhr in non_human_resources if nhr not in high_sensitive_resources]
    if row["tier"] == "Semi-autonomous":
      non_human_resources = [nhr for nhr in non_human_resources if nhr not in very_high_sensitive_resources]
    generate_grant_list(non_human_resources, "Engineering", row["entity_id"])

In [0]:
print(new_grant_list)

In [0]:
from pyspark.sql import functions as F

grants_df = spark.read.table("entity_risk_platform.seed_data.access_grants")
entities_df = spark.read.table("entity_risk_platform.seed_data.entities")
resources_df = spark.read.table("entity_risk_platform.seed_data.resources")

# 1. Basic shape
print("Total grants:", grants_df.count())
print("Distinct entities with grants:", grants_df.select("entity_id").distinct().count())

# 2. No duplicate grant_id
dup_grant_ids = grants_df.groupBy("grant_id").count().filter(F.col("count") > 1)
print("Duplicate grant_ids:", dup_grant_ids.count())

# 3. Orphan checks — every entity_id and resource_id should resolve
orphan_entities = grants_df.join(entities_df, "entity_id", "left_anti")
print("Grants with unknown entity_id:", orphan_entities.count())

orphan_resources = grants_df.join(resources_df, "resource_id", "left_anti")
print("Grants with unknown resource_id:", orphan_resources.count())

# 4. granted_by should be either 'SYSTEM' or a real entity_id
grants_with_res = grants_df.join(resources_df, "resource_id").join(entities_df, "entity_id")

bad_system_labels = grants_with_res.filter(
    (F.col("resource_sensitivity") != "high") & (F.col("granted_by") != "SYSTEM")
)
print("Low/medium grants NOT granted by SYSTEM:", bad_system_labels.count())

bad_expiry_low = grants_with_res.filter(
    (F.col("resource_sensitivity") != "high") & (F.col("expires_at").isNotNull())
)
print("Low/medium grants WITH an expiry (should be null):", bad_expiry_low.count())

high_grants = grants_with_res.filter(F.col("resource_sensitivity") == "high")

bad_no_expiry_high = high_grants.filter(F.col("expires_at").isNull())
print("High-sensitivity grants MISSING an expiry:", bad_no_expiry_high.count())

bad_system_high = high_grants.filter(F.col("granted_by") == "SYSTEM")
print("High-sensitivity grants granted by SYSTEM (should be a manager, or SYSTEM only if no manager existed):", bad_system_high.count())

# 5. granted_by, when not SYSTEM, must actually be a Manager-tier entity
managers_df = entities_df.filter(F.col("tier") == "Manager").select(F.col("entity_id").alias("granted_by"))
grantor_check = grants_df.filter(F.col("granted_by") != "SYSTEM").join(managers_df, "granted_by", "left_anti")
print("Grants where granted_by is NOT SYSTEM and NOT a Manager:", grantor_check.count())

# 6. Tier cap enforcement — no capped entity should have a high-sensitivity grant it shouldn't
very_high = ["admin_credentials_vault", "security_audit_logs", "legal_case_files"]

junior_violations = grants_with_res.filter(
    (F.col("entity_type") == "human") & (F.col("tier") == "Junior") & (F.col("resource_sensitivity") == "high")
)
print("Junior humans with a high-sensitivity grant (should be 0):", junior_violations.count())

fully_auto_violations = grants_with_res.filter(
    (F.col("entity_type").isin("service_account", "agent")) & (F.col("tier") == "Fully-autonomous") & (F.col("resource_sensitivity") == "high")
)
print("Fully-autonomous entities with a high-sensitivity grant (should be 0):", fully_auto_violations.count())

semi_auto_violations = grants_with_res.filter(
    (F.col("entity_type").isin("service_account", "agent")) & (F.col("tier") == "Semi-autonomous") & (F.col("resource_name").isin(very_high))
)
print("Semi-autonomous entities with one of the 3 excluded resources (should be 0):", semi_auto_violations.count())

# 7. Spot-check row counts per entity — every entity should have a grant count matching its capped candidate set size
grants_per_entity = grants_df.groupBy("entity_id").count()
print("\nSample of grants-per-entity distribution:")
grants_per_entity.groupBy("count").count().orderBy("count").show()

In [0]:
entities_df.join(
    grants_df.select("entity_id").distinct(), "entity_id", "left_anti"
).show()

In [0]:
grants_with_res.filter(
    (F.col("resource_sensitivity") == "high") & (F.col("granted_by") == "SYSTEM")
).select("entity_id", "role", "tier", "resource_name").show()